# 🗣️ Módulo 12 — Orchestration: Group Chat

> **Objetivo:** Coordinar una conversación grupal entre varios agentes con un orquestador que decide quién habla a continuación.

📚 **Doc oficial:** [Group Chat Orchestration (Python)](https://learn.microsoft.com/en-us/agent-framework/workflows/orchestrations/group-chat?pivots=programming-language-python)

## ¿Qué es Group Chat Orchestration?

Topología en **estrella** — todos los agentes hablan a través de un orquestador central que
decide qué agente toma el turno siguiente, basándose en una función de selección o en otro agente.

```
              ┌───────────┐
              │Orchestrator│
              └─┬───┬───┬─┘
        ┌───────┘   │   └────────┐
   ┌────▼───┐   ┌───▼───┐   ┌───▼────┐
   │ Agent A│   │Agent B│   │ Agent C│
   └────────┘   └───────┘   └────────┘
```

Ideal para:
- **Refinamiento iterativo** — round-robin de revisores
- **Colaboración multi-perspectiva** — Researcher + Writer + Critic
- Debates / brainstorming estructurado

## Selectores disponibles

| Selector | API | Cuándo usar |
|----------|-----|-------------|
| Round-robin | `selection_func=round_robin_selector` | Todos hablan por turno |
| Custom Python | `selection_func=lambda state: ...` | Lógica determinista basada en estado |
| Agent-based | `orchestrator_agent=Agent(...)` | LLM decide quién habla (con contexto) |


## ⚙️ Setup

In [ ]:
# Aseguramos que la raíz del proyecto esté en sys.path para importar `helpers`
import sys
from pathlib import Path

_ROOT = Path.cwd()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from agent_framework import Agent
from helpers.config import create_chat_client
print("✅ Helpers cargados.")


## 1️⃣ Round-robin selector

El selector más simple — recorre los participantes en orden. Útil cuando todos deben
contribuir por igual.


In [ ]:
from agent_framework.orchestrations import GroupChatBuilder, GroupChatState

client = create_chat_client()

researcher = Agent(
    client=client,
    name="Researcher",
    description="Recopila información relevante de fondo.",
    instructions="Recopila datos concisos que ayuden a responder la pregunta. Sé breve y factual. Responde siempre en español.",
    require_per_service_call_history_persistence=True,
)

writer = Agent(
    client=client,
    name="Writer",
    description="Sintetiza respuestas pulidas usando la información recopilada.",
    instructions="Compón respuestas claras y estructuradas usando las notas proporcionadas. Sé completo. Responde siempre en español.",
    require_per_service_call_history_persistence=True,
)


def round_robin_selector(state: GroupChatState) -> str:
    """Selecciona al siguiente speaker en round-robin."""
    names = list(state.participants.keys())
    return names[state.current_round % len(names)]


workflow = GroupChatBuilder(
    participants=[researcher, writer],
    termination_condition=lambda conv: sum(1 for m in conv if m.role == "assistant") >= 4,
    selection_func=round_robin_selector,
    intermediate_output_from=[researcher, writer],
).build()

print("✅ Group chat configurado con round-robin (máx 4 mensajes de agentes)")

## 2️⃣ Ejecutar y observar la conversación

Activa `stream=True` para ver cada actualización a medida que llega.
El output terminal es un `AgentResponse` con el resumen del orquestador.


In [ ]:
from agent_framework import AgentResponse, AgentResponseUpdate

task = "¿Cuáles son los beneficios clave de async/await en Python?"

print(f"📨 Task: {task}\n{'=' * 60}\n")

last_author = None

async for event in workflow.run(task, stream=True):
    data = event.data
    if isinstance(data, AgentResponseUpdate) and data.text:
        if data.author_name != last_author:
            if last_author is not None:
                print("\n")
            print(f"🤖 [{data.author_name}]: ", end="", flush=True)
            last_author = data.author_name
        print(data.text, end="", flush=True)

print(f"\n\n{'=' * 60}\n✅ Conversación completada.")

## 3️⃣ Selector custom basado en contenido

`GroupChatState.conversation` te da acceso a todos los mensajes hasta ahora.
Puedes implementar lógica inteligente — por ejemplo, que el `Researcher` investigue
primero y luego el `Writer` sintetice la respuesta final.

In [ ]:
def content_based_selector(state: GroupChatState) -> str:
    """Researcher habla primero, Writer sintetiza después. Alterna si Researcher ya contribuyó."""
    assistant_msgs = [m for m in state.conversation if m.role == "assistant"]

    if len(assistant_msgs) == 0:
        return "Researcher"  # Primera ronda: investigar

    last = assistant_msgs[-1]
    # Después de que Researcher habla, Writer sintetiza
    if last.author_name == "Researcher":
        return "Writer"
    # Después de Writer, Researcher puede añadir más datos
    return "Researcher"


workflow = GroupChatBuilder(
    participants=[researcher, writer],
    termination_condition=lambda conv: sum(1 for m in conv if m.role == "assistant") >= 4,
    selection_func=content_based_selector,
    intermediate_output_from=[researcher, writer],
).build()

print("📨 Task: Explicar los transformadores de mónadas\n")

last_author = None
async for event in workflow.run("Explica brevemente qué son los transformadores de mónadas en programación funcional.", stream=True):
    data = event.data
    if isinstance(data, AgentResponseUpdate) and data.text:
        if data.author_name != last_author:
            if last_author is not None:
                print("\n")
            print(f"🤖 [{data.author_name}]: ", end="", flush=True)
            last_author = data.author_name
        print(data.text, end="", flush=True)

print("\n\n✅ Selector basado en contenido completado.")

## 4️⃣ Orquestador basado en agente (LLM)

Para conversaciones más complejas, deja que **otro agente** decida quién habla a continuación.
El orquestador es un `Agent` completo con sus propias instructions y tools.


In [ ]:
orchestrator_agent = Agent(
    client=client,
    name="Orchestrator",
    description="Coordina la colaboración multi-agente seleccionando quién habla",
    instructions=(
        "Coordinas una conversación de equipo para resolver la tarea del usuario.\n\n"
        "Directrices:\n"
        "- Empieza con Researcher para recopilar información\n"
        "- Luego Writer sintetiza la respuesta final\n"
        "- Alterna entre ellos — nunca selecciones al mismo agente dos veces seguidas\n"
        "- Solo termina cuando ambos hayan contribuido significativamente\n"
        "- Responde siempre en español"
    ),
    require_per_service_call_history_persistence=True,
)

workflow = GroupChatBuilder(
    participants=[researcher, writer],
    termination_condition=lambda msgs: sum(1 for m in msgs if m.role == "assistant") >= 4,
    orchestrator_agent=orchestrator_agent,
    intermediate_output_from=[researcher, writer],
).build()

print("📨 Task: SQLite vs PostgreSQL para una startup SaaS\n")

turns: list[tuple[str, str]] = []
current_author = None
current_text = []

async for event in workflow.run(
    "¿Cuáles son las ventajas y desventajas de usar SQLite vs PostgreSQL para una startup SaaS?",
    stream=True,
):
    data = event.data
    if isinstance(data, AgentResponseUpdate) and data.text:
        if data.author_name != current_author:
            if current_author and current_text:
                turns.append((current_author, "".join(current_text)))
            current_author = data.author_name
            current_text = []
        current_text.append(data.text)

if current_author and current_text:
    turns.append((current_author, "".join(current_text)))

for i, (author, text) in enumerate(turns, 1):
    preview = text[:200] + "..." if len(text) > 200 else text
    print(f"🤖 Turno {i} [{author}]: {preview}\n")

print(f"✅ Conversación completada — {len(turns)} turnos de agentes.")

## 5️⃣ Junta de equipo de desarrollo — Group Chat con moderador IA

Simulamos una **junta técnica real** con 4 agentes especializados — un **Arquitecto**, un **Desarrollador**,
un **DBA** y un **Project Manager** — que discuten un requerimiento y llegan a una solución.

> 💡 Este patrón replica la orquestación `EquipoDesarrolloAI` de [AF-WebChat](../../02-AFWebChat/),
> donde un moderador IA decide quién habla según el contexto.

```
           ┌──────────────┐
           │  Moderador   │
           └─┬──┬──┬──┬───┘
      ┌──────┘  │  │  └──────┐
 ┌────▼───┐ ┌──▼──▼──┐ ┌────▼───┐
 │Arquit. │ │Dev  DBA│ │  PM    │
 └────────┘ └────────┘ └────────┘
```

In [ ]:
# --- Equipo de desarrollo: 4 agentes que discuten un requerimiento técnico ---
# Basado en los agentes de 02-AFWebChat/Agents/Workflow/

arquitecto = Agent(
    client=client,
    name="Arquitecto",
    description="Arquitecto de software que diseña la solución a alto nivel, define patrones y toma decisiones técnicas.",
    instructions=(
        "Eres un Arquitecto de Software / Solutions Architect con experiencia en sistemas empresariales.\n"
        "Estás en una junta técnica junto con un Desarrollador, un PM y un DBA.\n\n"
        "Tu perspectiva es SIEMPRE la de diseño y visión a largo plazo:\n"
        "- Propones la arquitectura: monolito vs microservicios, cloud, event-driven, etc.\n"
        "- Defines patrones de diseño (CQRS, Event Sourcing, DDD)\n"
        "- Evalúas trade-offs: rendimiento vs complejidad, costo vs escalabilidad\n"
        "- Piensas en NFRs: seguridad, disponibilidad, mantenibilidad\n\n"
        "REGLA: Responde DIRECTAMENTE a lo que dijeron los otros en esta conversación.\n"
        "Si el Desarrollador propone algo, opina si escala. Si el PM pide algo, negocia.\n"
        "Si el DBA sugiere un modelo, evalúa si encaja con la arquitectura.\n"
        "Usa diagramas ASCII cuando ayude. Responde en español. Máx 4-5 oraciones.\n"
        "Emojis: 🏗️ arquitectura, ⚡ rendimiento, 🔐 seguridad, ⚖️ trade-off."
    ),
)

desarrollador = Agent(
    client=client,
    name="Desarrollador",
    description="Desarrollador senior que propone soluciones de código, evalúa implementaciones y sugiere mejores prácticas.",
    instructions=(
        "Eres un Desarrollador Full-Stack Senior con 10+ años de experiencia.\n"
        "Estás en una junta técnica junto con un Arquitecto, un PM y un DBA.\n\n"
        "Tu perspectiva es SIEMPRE la de implementación práctica:\n"
        "- Propones cómo construirlo: frameworks, librerías, patrones\n"
        "- Identificas complejidades técnicas y deuda técnica potencial\n"
        "- Estimas esfuerzo real de desarrollo (story points o días)\n"
        "- Sugieres cómo dividir el trabajo en tareas/PRs\n"
        "- Adviertes sobre edge cases y bugs potenciales\n\n"
        "REGLA: Responde DIRECTAMENTE a lo que dijeron los otros en esta conversación.\n"
        "Si no estás de acuerdo con el Arquitecto, dilo directo. Si el PM pide algo irreal, advierte.\n"
        "Si el DBA propone algo, evalúa cómo afecta al código.\n"
        "Responde en español. Máx 4-5 oraciones.\n"
        "Emojis: ✅ viable, ⚠️ riesgo, 🔴 blocker, 💡 idea, 🤔 duda."
    ),
)

dba = Agent(
    client=client,
    name="DBA",
    description="DBA que diseña modelos de datos, optimiza queries y define estrategias de persistencia.",
    instructions=(
        "Eres un DBA / Data Engineer senior con experiencia en SQL Server, Azure SQL, PostgreSQL y NoSQL.\n"
        "Estás en una junta técnica junto con un Desarrollador, un Arquitecto y un PM.\n\n"
        "Tu perspectiva es SIEMPRE la de los datos:\n"
        "- Propones el modelo de datos: entidades, relaciones, normalización\n"
        "- Recomiendas el motor de BD apropiado (SQL vs NoSQL vs híbrido)\n"
        "- Adviertes sobre problemas de rendimiento y cuellos de botella\n"
        "- Evalúas si necesitan cache (Redis), search (Elasticsearch), etc.\n\n"
        "REGLA: Responde DIRECTAMENTE a lo que dijeron los otros en esta conversación.\n"
        "Si el Arquitecto propone algo que mate el rendimiento de la BD, dilo sin rodeos.\n"
        "Si el Desarrollador sugiere queries complejas, propón alternativas.\n"
        "Usa tablas cuando propongas modelos. Responde en español. Máx 4-5 oraciones.\n"
        "Emojis: 🗄️ estructura, ⚡ performance, 📊 volumen, ⚠️ riesgo."
    ),
)

pm = Agent(
    client=client,
    name="ProjectManager",
    description="Project Manager que coordina alcance, tiempos, riesgos y prioridades del equipo.",
    instructions=(
        "Eres un Project Manager / Scrum Master con certificación PMP.\n"
        "Estás en una junta técnica junto con un Desarrollador, un Arquitecto y un DBA.\n\n"
        "Tu perspectiva es SIEMPRE la de gestión y entrega:\n"
        "- Preguntas por alcance, tiempos y dependencias\n"
        "- Priorizas features por valor de negocio vs esfuerzo\n"
        "- Identificas riesgos y propones mitigaciones\n"
        "- Defines milestones y entregables claros\n"
        "- Si el equipo se pierde en detalles, regresa al objetivo de negocio\n\n"
        "REGLA: Responde DIRECTAMENTE a lo que dijeron los otros en esta conversación.\n"
        "Si algo afecta el timeline, advierte inmediatamente.\n"
        "Si el equipo discute sin avanzar, propón un plan de acción concreto.\n"
        "Responde en español. Máx 4-5 oraciones.\n"
        "Emojis: ⏱️ timeline, 🎯 prioridad, ⚠️ riesgo, ✅ próximos pasos."
    ),
)

# --- Moderador que simula un Tech Lead dirigiendo la junta ---

tech_lead = Agent(
    client=client,
    name="TechLead",
    description="Tech Lead que modera la junta técnica y asegura que el equipo llegue a una solución",
    instructions=(
        "Eres el Tech Lead moderando una junta técnica entre Arquitecto, Desarrollador, DBA y ProjectManager.\n\n"
        "Reglas para seleccionar quién habla:\n"
        "- Empieza con Arquitecto para la visión general\n"
        "- Luego Desarrollador para evaluar viabilidad de implementación\n"
        "- DBA para el modelo de datos y performance\n"
        "- ProjectManager para alcance, tiempos y plan\n"
        "- Después, deja que se retroalimenten: si el Arquitecto propuso algo, "
        "el Desarrollador debe opinar si es viable, el DBA si los datos aguantan\n"
        "- Fomenta el debate: si todos están de acuerdo, pide que busquen riesgos\n"
        "- Nunca el mismo agente dos veces seguidas\n"
        "- Al final, el PM debe cerrar con el plan de acción"
    ),
)

junta = GroupChatBuilder(
    participants=[arquitecto, desarrollador, dba, pm],
    termination_condition=lambda conv: sum(1 for m in conv if m.role == "assistant") >= 8,
    orchestrator_agent=tech_lead,
    intermediate_output_from=[arquitecto, desarrollador, dba, pm],
).build()

# --- Ejecución con streaming en tiempo real ---

emojis = {"Arquitecto": "🏗️", "Desarrollador": "👨‍💻", "DBA": "🗄️", "ProjectManager": "📋"}

requerimiento = (
    "El cliente necesita un sistema de notificaciones en tiempo real para su plataforma SaaS. "
    "Debe soportar 50,000 usuarios concurrentes, notificaciones push (web + móvil), "
    "email y SMS. Necesitan un historial de notificaciones con búsqueda, "
    "y la capacidad de que los usuarios configuren sus preferencias de canal. "
    "Presupuesto moderado, deadline en 3 meses."
)

print(f"📨 REQUERIMIENTO:\n{requerimiento}\n{'═' * 60}\n")

last_author = None
turns = 0

async for event in junta.run(requerimiento, stream=True):
    data = event.data
    if isinstance(data, AgentResponseUpdate) and data.text:
        if data.author_name in emojis:
            if data.author_name != last_author:
                if last_author is not None:
                    turns += 1
                    print(f"\n{'─' * 60}\n")
                emoji = emojis[data.author_name]
                print(f"{emoji} [{data.author_name}]: ", end="", flush=True)
                last_author = data.author_name
            print(data.text, end="", flush=True)

if last_author:
    turns += 1

print(f"\n\n{'═' * 60}")
print(f"✅ Junta completada — {turns} intervenciones del equipo.")

## 🎯 Resumen

| Capacidad | API |
|-----------|-----|
| Round-robin | `selection_func=round_robin_selector` (con `GroupChatState`) |
| Selector custom | `selection_func=lambda state: "AgentName"` |
| Orquestador LLM | `orchestrator_agent=Agent(...)` |
| Condición de terminación | `termination_condition=lambda conv: len(conv) >= N` |
| Outputs intermedios | `intermediate_output_from=[a, b]` |
| Streaming | `async for event in workflow.run(task, stream=True):` |

## 📊 Comparación de patrones de orquestación

| Patrón | Cuándo usar |
|--------|-------------|
| **Sequential** (mod 09) | Pipeline lineal con etapas claras |
| **Concurrent** (mod 10) | Múltiples perspectivas en paralelo |
| **Handoff** (mod 11) | Delegación dinámica entre especialistas |
| **Group Chat** (mod 12) | Conversación colaborativa con turnos |

**Siguiente módulo →** [Módulo 13 — Workflows: Ejecutores (low-level)](./13_workflows_executors.ipynb)
